In [1]:
from __future__ import annotations

from collections.abc import Callable

import numpy as np
from numpy.typing import NDArray


FloatArray = NDArray[np.float64]
ComplexArray = NDArray[np.complex128]


def bloch_laplacian_eigenvalues(
    bloch_wavevector: FloatArray,
    reciprocal_lattice_vectors: FloatArray,
) -> FloatArray:
    """
    Compute the eigenvalues of the Bloch Laplacian in a plane-wave basis.

    The plane-wave basis functions are

        exp(i G · r),

    where G is a reciprocal-lattice vector. For a fixed Bloch
    wavevector k, the Bloch Laplacian acts according to

        (∇ + i k)^2 exp(i G · r)
        =
        -|k + G|^2 exp(i G · r).

    Parameters
    ----------
    bloch_wavevector:
        Bloch wavevector k with shape ``(dimension,)``.

    reciprocal_lattice_vectors:
        Reciprocal-lattice vectors G with shape
        ``(number_of_plane_waves, dimension)``.

    Returns
    -------
    FloatArray
        Bloch-Laplacian eigenvalues ``-|k + G|^2`` with shape
        ``(number_of_plane_waves,)``.
    """
    bloch_wavevector = np.asarray(
        bloch_wavevector,
        dtype=np.float64,
    )

    reciprocal_lattice_vectors = np.asarray(
        reciprocal_lattice_vectors,
        dtype=np.float64,
    )

    if bloch_wavevector.ndim != 1:
        raise ValueError(
            "bloch_wavevector must have shape (dimension,)."
        )

    if reciprocal_lattice_vectors.ndim != 2:
        raise ValueError(
            "reciprocal_lattice_vectors must have shape "
            "(number_of_plane_waves, dimension)."
        )

    if (
        reciprocal_lattice_vectors.shape[1]
        != bloch_wavevector.shape[0]
    ):
        raise ValueError(
            "bloch_wavevector and reciprocal_lattice_vectors "
            "must have the same spatial dimension."
        )

    shifted_wavevectors = (
        reciprocal_lattice_vectors
        + bloch_wavevector[None, :]
    )

    return -np.sum(
        shifted_wavevectors**2,
        axis=1,
    )


def bloch_laplacian_matrix(
    bloch_wavevector: FloatArray,
    reciprocal_lattice_vectors: FloatArray,
) -> ComplexArray:
    """
    Construct the Bloch-Laplacian matrix in a plane-wave basis.

    The matrix elements are

        L_G'G(k)
        =
        -|k + G|^2 δ_G'G.

    Parameters
    ----------
    bloch_wavevector:
        Bloch wavevector k with shape ``(dimension,)``.

    reciprocal_lattice_vectors:
        Reciprocal-lattice vectors G with shape
        ``(number_of_plane_waves, dimension)``.

    Returns
    -------
    ComplexArray
        Diagonal Bloch-Laplacian matrix with shape
        ``(number_of_plane_waves, number_of_plane_waves)``.
    """
    eigenvalues = bloch_laplacian_eigenvalues(
        bloch_wavevector,
        reciprocal_lattice_vectors,
    )

    return np.diag(
        eigenvalues.astype(np.complex128)
    )


def bloch_kinetic_energy_matrix(
    bloch_wavevector: FloatArray,
    reciprocal_lattice_vectors: FloatArray,
    *,
    hbar: float = 1.0,
    mass: float = 1.0,
) -> ComplexArray:
    """
    Construct the kinetic-energy matrix for a fixed Bloch wavevector.

    The kinetic-energy operator is

        T(k)
        =
        -(hbar^2 / 2m) (∇ + i k)^2.

    Its plane-wave matrix elements are

        T_G'G(k)
        =
        (hbar^2 / 2m)
        |k + G|^2 δ_G'G.

    Parameters
    ----------
    bloch_wavevector:
        Bloch wavevector k with shape ``(dimension,)``.

    reciprocal_lattice_vectors:
        Reciprocal-lattice vectors G with shape
        ``(number_of_plane_waves, dimension)``.

    hbar:
        Reduced Planck constant.

    mass:
        Particle mass.

    Returns
    -------
    ComplexArray
        Kinetic-energy matrix.
    """
    if hbar <= 0.0:
        raise ValueError("hbar must be positive.")

    if mass <= 0.0:
        raise ValueError("mass must be positive.")

    laplacian = bloch_laplacian_matrix(
        bloch_wavevector,
        reciprocal_lattice_vectors,
    )

    return -(hbar**2 / (2.0 * mass)) * laplacian


def periodic_potential_matrix(
    reciprocal_lattice_vectors: FloatArray,
    fourier_coefficient: Callable[[FloatArray], complex],
) -> ComplexArray:
    """
    Construct the periodic-potential matrix in a plane-wave basis.

    The matrix elements are

        V_G'G
        =
        V_(G' - G),

    where ``V_q`` is the Fourier coefficient of the periodic
    potential at reciprocal-lattice vector q.

    Parameters
    ----------
    reciprocal_lattice_vectors:
        Reciprocal-lattice vectors G with shape
        ``(number_of_plane_waves, dimension)``.

    fourier_coefficient:
        Function that evaluates the Fourier coefficient ``V_q``.

    Returns
    -------
    ComplexArray
        Periodic-potential matrix.
    """
    reciprocal_lattice_vectors = np.asarray(
        reciprocal_lattice_vectors,
        dtype=np.float64,
    )

    if reciprocal_lattice_vectors.ndim != 2:
        raise ValueError(
            "reciprocal_lattice_vectors must have shape "
            "(number_of_plane_waves, dimension)."
        )

    number_of_plane_waves = len(
        reciprocal_lattice_vectors
    )

    potential = np.zeros(
        (
            number_of_plane_waves,
            number_of_plane_waves,
        ),
        dtype=np.complex128,
    )

    for row, reciprocal_vector_prime in enumerate(
        reciprocal_lattice_vectors
    ):
        for column, reciprocal_vector in enumerate(
            reciprocal_lattice_vectors
        ):
            transferred_wavevector = (
                reciprocal_vector_prime
                - reciprocal_vector
            )

            potential[row, column] = (
                fourier_coefficient(
                    transferred_wavevector
                )
            )

    return potential


def bloch_hamiltonian_matrix(
    bloch_wavevector: FloatArray,
    reciprocal_lattice_vectors: FloatArray,
    fourier_coefficient: Callable[[FloatArray], complex],
    *,
    hbar: float = 1.0,
    mass: float = 1.0,
) -> ComplexArray:
    """
    Construct the plane-wave Bloch Hamiltonian.

    The matrix elements are

        H_G'G(k)
        =
        (hbar^2 / 2m)
        |k + G|^2 δ_G'G
        +
        V_(G' - G).

    Parameters
    ----------
    bloch_wavevector:
        Bloch wavevector k with shape ``(dimension,)``.

    reciprocal_lattice_vectors:
        Reciprocal-lattice vectors G with shape
        ``(number_of_plane_waves, dimension)``.

    fourier_coefficient:
        Function that evaluates ``V_q``.

    hbar:
        Reduced Planck constant.

    mass:
        Particle mass.

    Returns
    -------
    ComplexArray
        Hermitian Bloch-Hamiltonian matrix.
    """
    kinetic_energy = bloch_kinetic_energy_matrix(
        bloch_wavevector,
        reciprocal_lattice_vectors,
        hbar=hbar,
        mass=mass,
    )

    potential_energy = periodic_potential_matrix(
        reciprocal_lattice_vectors,
        fourier_coefficient,
    )

    hamiltonian = kinetic_energy + potential_energy

    hermiticity_error = np.max(
        np.abs(
            hamiltonian
            - hamiltonian.conj().T
        )
    )

    if hermiticity_error > 1.0e-10:
        raise ValueError(
            "The Bloch Hamiltonian is not Hermitian. "
            f"Maximum error: {hermiticity_error:.3e}."
        )

    return hamiltonian


def solve_bloch_hamiltonian(
    bloch_wavevector: FloatArray,
    reciprocal_lattice_vectors: FloatArray,
    fourier_coefficient: Callable[[FloatArray], complex],
    *,
    number_of_bands: int | None = None,
    hbar: float = 1.0,
    mass: float = 1.0,
) -> tuple[FloatArray, ComplexArray]:
    """
    Solve the Bloch eigenvalue problem at one wavevector k.

    Returns
    -------
    energies:
        Band energies ``E_n(k)``.

    coefficients:
        Plane-wave coefficients. Column ``n`` contains the
        coefficients ``c_nk(G)`` of eigenstate ``n``.
    """
    hamiltonian = bloch_hamiltonian_matrix(
        bloch_wavevector,
        reciprocal_lattice_vectors,
        fourier_coefficient,
        hbar=hbar,
        mass=mass,
    )

    energies, coefficients = np.linalg.eigh(
        hamiltonian
    )

    if number_of_bands is None:
        return energies, coefficients

    if not 1 <= number_of_bands <= len(energies):
        raise ValueError(
            "number_of_bands must be between 1 and the "
            "number of plane waves."
        )

    return (
        energies[:number_of_bands],
        coefficients[:, :number_of_bands],
    )